In [9]:
import numpy as np
import matplotlib.pyplot as plt

import sympy as sp

In [10]:
try:
    import casadi
except:
    !pip install casadi
    import casadi

try:
    import do_mpc
except:
    !pip install do_mpc
    import do_mpc

try:
    import pybounds
except:
    #!pip install pybounds
    !pip install git+https://github.com/vanbreugel-lab/pybounds
    import pybounds

### Import plotting utilities and planar drone locally or from github

In [11]:
import sys
import requests
import importlib

def import_local_or_github(package_name, function_name=None, directory=None, giturl=None):
    # Import functions directly from github
    # Important: note that we use raw.githubusercontent.com, not github.com

    try: # to find the file locally
        if directory is not None:
            if directory not in sys.path:
                sys.path.append(directory)

        package = importlib.import_module(package_name)
        if function_name is not None:
            function = getattr(package, function_name)
            return function
        else:
            return package

    except: # get the file from github
        if giturl is None:
            giturl = 'https://raw.githubusercontent.com/florisvb/Nonlinear_and_Data_Driven_Estimation/main/Utility/' + str(package_name) + '.py'

        r = requests.get(giturl)
        print('Fetching from: ')
        print(r)

        # Store the file to the colab working directory
        with open(package_name+'.py', 'w') as f:
            f.write(r.text)
        f.close()

        # import the function we want from that file
        package = importlib.import_module(package_name)
        if function_name is not None:
            function = getattr(package , function_name)
            return function
        else:
            return package

planar_drone = import_local_or_github('planar_drone', directory='../Utility')
plot_tme = import_local_or_github('plot_utility', 'plot_tme', directory='../Utility')
dynamics_utility = import_local_or_github('dynamics_utility', directory='../Utility')

#### Given the planar drone dynamics

$
\mathbf{\dot{x}} = \mathbf{f}(\mathbf{x},\mathbf{u}) =
\frac{d}{dt}
\begin{bmatrix}
\bbox[yellow]{\theta} \\[0.3em]
\bbox[yellow]{\dot{\theta}} \\[0.3em]
\bbox[yellow]{x} \\[0.3em]
\bbox[yellow]{\dot{x}} \\[0.3em]
\bbox[yellow]{z} \\[0.3em]
\bbox[yellow]{\dot{z}} \\[0.3em]
\bbox[pink]{k}
\end{bmatrix} =
\overset{f_0}{\begin{bmatrix}
\bbox[yellow]{\dot{\theta}} \\[0.3em]
0 \\[0.3em]
\bbox[yellow]{\dot{x}} \\[0.3em]
0 \\[0.3em]
\bbox[yellow]{\dot{z}} \\[0.3em]
-\bbox[lightblue]{g} \\[0.3em]
0
\end{bmatrix}} +
\overset{f_1}{\begin{bmatrix}
0 \\[0.3em]
\bbox[lightblue]{l}\bbox[pink]{k}/\bbox[lightblue]{I_{yy}} \\[0.3em]
0 \\[0.3em]
0 \\[0.3em]
0 \\[0.3em]
0 \\[0.3em]
0
\end{bmatrix}} \bbox[lightgreen]{j_1} +
\overset{f_2}{\begin{bmatrix}
0 \\[0.3em]
0 \\[0.3em]
0 \\[0.3em]
-\bbox[pink]{k}\bbox[yellow]{\sin\theta}/\bbox[lightblue]{m} \\[0.3em]
0 \\[0.3em]
\bbox[pink]{k}\bbox[yellow]{\cos\theta}/\bbox[lightblue]{m}  \\[0.3em]
0
\end{bmatrix}} \bbox[lightgreen]{j_2}
$

#### Consider the following sensor combinations:

$
\mathbf{y_a} = \mathbf{{h_a}}(\mathbf{{x}}, \mathbf{{u}}) =
\begin{bmatrix}
\bbox[yellow]{\theta} \\[0.3em]
\bbox[yellow]{x} \\[0.3em]
\bbox[yellow]{z} \\[0.3em]
\bbox[pink]{k}
\end{bmatrix}
$

$
\mathbf{y_b} = \mathbf{{h_b}}(\mathbf{{x}}, \mathbf{{u}}) =
\begin{bmatrix}
\bbox[yellow]{\dot{x}}/\bbox[yellow]{z} \\[0.3em]
\bbox[yellow]{\theta} \\[0.3em]
\bbox[pink]{k}
\end{bmatrix}
$

$
\mathbf{y_c} = \mathbf{{h_c}}(\mathbf{{x}}, \mathbf{{u}}) =
\begin{bmatrix}
\bbox[yellow]{\dot{x}}/\bbox[yellow]{z} \\[0.3em]
\bbox[yellow]{\theta} \\[0.3em]
\bbox[yellow]{\dot{\theta}}   \\[0.3em]
\ddot{x} = -\bbox[pink]{k} \sin(\bbox[yellow]{\theta}) \bbox[lightgreen]{j_1} / \bbox[lightblue]{m}  \\[0.3em]
\ddot{z} = -\bbox[lightblue]{g} / \bbox[lightblue]{m} + \bbox[pink]{k} \cos(\bbox[yellow]{\theta}) \bbox[lightgreen]{j_2}/ \bbox[lightblue]{m}
\end{bmatrix}
$

In [12]:
f = planar_drone.F().f
h_a = planar_drone.H('h_gps').h
h_b = planar_drone.H('h_camera_theta_k').h
h_c = planar_drone.H('h_camera_imu').h

### Linearize the dynamics and measurement function

We need linearized dynamics of the form:

$
\mathbf{x_{k+1}} = A\mathbf{x_k} + B\mathbf{u_k}
$

$
\mathbf{y_k} = C\mathbf{x_k} + D\mathbf{u_k}
$

We will do that numerically using the same functions as before.

In [13]:
jacobian_numerical = dynamics_utility.jacobian_numerical
rk4_discretize = dynamics_utility.rk4_discretize

In [14]:
# initial condition for planar drone in hover mode

# x = [theta, thetadot, x, xdot, z, zdot, k]
x0 = np.array([0, 0, 0, 0, 1, 0, 1])
u0 = np.array([0, 0])

In [15]:
def f_discrete(x, u):
    dt = 0.1
    return rk4_discretize(f, x, u, dt)

In [16]:
A, B = jacobian_numerical(f_discrete, x0, u0)

In [17]:
C_a, D_a = jacobian_numerical(h_a, x0, u0)
C_b, D_b = jacobian_numerical(h_b, x0, u0)
C_c, D_c = jacobian_numerical(h_c, x0, u0)

# Exercises

1. Determine the rank of the observability matrix for each measurement option. Which ones are observable?
2. Determine the observability Gramian for each measurement option:
  * Analyze the eigenvalues/eigenvectors to determine the most observable state combinations
  * Analyze the matrix inverse of the Gramian to determien the most observable individual states

3. Do your results agree with your Kalman filter from the prior lesson?  

In [33]:
Observability1 = [C_a, C_a@A, C_a@A@A, C_a@A@A@A, C_a@A@A@A@A]
Observability1

Observability2 = [C_b, C_b@A, C_b@A@A, C_b@A@A@A, C_b@A@A@A@A]
Observability2

Observability3 = [C_c, C_c@A, C_c@A@A, C_c@A@A@A, C_c@A@A@A@A]
Observability3

[array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00],
        [ 1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00],
        [ 0.00000000e+00,  1.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00],
        [-9.99999833e-01,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00],
        [-4.99999958e-04,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          1.00000000e+00]]),
 array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00],
        [ 1.00000000e+00,  1.00000000e-01,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+

In [34]:
Observability1_matrix = np.vstack(Observability1)
Observability2_matrix = np.vstack(Observability2)
Observability3_matrix = np.vstack(Observability3)

rank1 = np.linalg.matrix_rank(Observability1_matrix)
rank2 = np.linalg.matrix_rank(Observability2_matrix)
rank3 = np.linalg.matrix_rank(Observability3_matrix)

print(f'Rank of Observability Matrix 1: {rank1}')
print(f'Rank of Observability Matrix 2: {rank2}')
print(f'Rank of Observability Matrix 3: {rank3}')

Rank of Observability Matrix 1: 7
Rank of Observability Matrix 2: 4
Rank of Observability Matrix 3: 4


In [38]:
Observability1_Gramian = Observability1_matrix.T @ Observability1_matrix
Observability2_Gramian = Observability2_matrix.T @ Observability2_matrix
Observability3_Gramian = Observability3_matrix.T @ Observability3_matrix

eigenvalues1, eigenvectors1 = np.linalg.eig(Observability1_Gramian)
eigenvalues2, eigenvectors2 = np.linalg.eig(Observability2_Gramian)
eigenvalues3, eigenvectors3 = np.linalg.eig(Observability3_Gramian)

print("Eigenvalues of Observability Gramian 1:", eigenvalues1)
print("Eigenvectors of Observability Gramian 1:", eigenvectors1)
print("\nEigenvalues of Observability Gramian 2:", eigenvalues2)
print("Eigenvectors of Observability Gramian 2:", eigenvectors2)
print("\nEigenvalues of Observability Gramian 3:", eigenvalues3)
print("Eigenvectors of Observability Gramian 3:", eigenvectors3)

Eigenvalues of Observability Gramian 1: [5.20391856 0.09608144 5.20391856 0.09608144 5.20391856 0.09608144
 5.        ]
Eigenvectors of Observability Gramian 1: [[ 0.97983535 -0.19980661  0.          0.          0.          0.
   0.        ]
 [ 0.19980661  0.97983535  0.          0.          0.          0.
   0.        ]
 [ 0.          0.          0.97983535 -0.19980661  0.          0.
   0.        ]
 [ 0.          0.          0.19980661  0.97983535  0.          0.
   0.        ]
 [ 0.          0.          0.          0.          0.97983535 -0.19980661
   0.        ]
 [ 0.          0.          0.          0.          0.19980661  0.97983535
   0.        ]
 [ 0.          0.          0.          0.          0.          0.
   1.        ]]

Eigenvalues of Observability Gramian 2: [5.20391856 0.09608144 0.         5.         0.         0.
 5.        ]
Eigenvectors of Observability Gramian 2: [[ 0.97983535 -0.19980661  0.          0.          0.          0.
   0.        ]
 [ 0.19980661  0.979

In [44]:
Observability1_Gramian.T
Observability2_Gramian.T
Observability3_Gramian.T

array([[ 9.99999958e+00,  1.99999992e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        -2.49999979e-03],
       [ 1.99999992e+00,  5.59999998e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        -4.99999958e-04],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         5.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [-2.49999979e-03, -4.99999958e-04,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         5.0000000